In [127]:
import pandas as pd
import numpy as np

In [141]:
df_hh=pd.read_csv('df_hh.csv')
print(len(df_hh))
df_hh.head()

13551


,title,hh_link,address,company,short_description,experience,salary
0,Разработчик,https://hh.ru/vacancy/131819443?query=%D1%80%D...,Москва,"Русская Медиагруппа, радиохолдинг",Опыт работы,Опыт 3-6 лет,NaN
1,Разработчик,https://hh.ru/vacancy/131986214?query=%D1%80%D...,Москва,Детский хоспис Дом с маяком,"Уверенное знание Python 3, базовых принципов О...",Опыт 1-3 года,"от 95 000 ₽ за месяц, на руки"
2,Backend-разработчик,https://hh.ru/vacancy/131274910?query=%D1%80%D...,Москва,ООО ГК СтиС,2+ года коммерческой разработки на PHP 7.4+. У...,Опыт 1-3 года,NaN
3,Разработчик ЦФТ,https://hh.ru/vacancy/132003073?query=%D1%80%D...,Москва,Bell Integrator,Опыт работы с АБС «ЦФТ-Банк» в качестве,Опыт 3-6 лет,NaN
4,Разработчик C++,https://hh.ru/vacancy/132001730?query=%D1%80%D...,Москва,ООО БУЛАТ,Опыт разработки или исправления/доработки внут...,Опыт 3-6 лет,NaN


# Удаляем лишний столбец

Ссылка на вакансию нам не нужна - удалим этот столбец.

In [142]:
df_hh=df_hh.drop(columns='hh_link')

# Удаляем дубли

In [143]:
df_hh=df_hh.drop_duplicates()
print(13551-len(df_hh),'дублей удалили')
print(len(df_hh),'строк осталось')

1972 дублей удалили
11579 строк осталось


Были дубликаты из-за метода сбора – при скрепинге разными запросами искались ИТ-вакансии (по разным словам), из-за чего возникли пересечения.

# Обогащаем датасет hh

## Добавляем признаки Регион и Страна

По городам присоединим регион и страну, воспользовавшись справочником из АПИ суперджоба.

In [144]:
df_hh['address'].unique()

<StringArray>
[                                       'Москва',
                         'Москва, р-н Басманный',
                      'Москва, р-н Дорогомилово',
                       'Москва, р-н Даниловский',
                    'Москва, р-н Красносельский',
                     'Москва, р-н Замоскворечье',
                       'Москва, р-н Пресненский',
                         'Москва, р-н Хамовники',
                       'Зеленоград, р-н Савёлки',
                             'деревня Румянцево',
 ...
                       'Калининградская область',
           'Витебск, улица Петруся Бровки, 50к7',
           'Махачкала, Хаджалмахинская улица, 1',
                 'Дмитров, Московская улица, 29',
            'Бишкек, улица Михаила Фрунзе, 533А',
 'Набережные Челны, Старосармановская улица, 29',
   'Ташкент, Яшнабадский район, улица Пунган, 7',
                  'Брест, проспект Машерова, 17',
                 'Подольск, Фабричный проезд, 2',
                    'Владивосто

Видим, что в столбце address не только город, но и р-н/улица. Оставим только город. Предварительно проверим пропуски.

In [145]:
df_hh[df_hh['address'].isna()]

,title,address,company,short_description,experience,salary
1752,Junior PHP разработчик,NaN,NaN,NaN,NaN,NaN


Один пропуск (пустая вакансия). Дропним эту строку.

In [146]:
df_hh=df_hh.drop(1752)
df_hh['address'].isna().sum()

np.int64(0)

In [147]:
df_hh['address']=df_hh['address'].apply(lambda x: x.split(',')[0])
df_hh['address'].unique()

<StringArray>
[                             'Москва',                          'Зеленоград',
                   'деревня Румянцево',                            'Тольятти',
     'посёлок городского типа Родники',                               'Химки',
                          'Никольское',                         'Красногорск',
                     'Санкт-Петербург',                           'г. Москва',
 ...
    'Черняховский муниципальный округ',                              'Латвия',
                  'посёлок Ольховатка',                             'Испания',
                           'Гвардейск',               'Бугрино (Ненецкий АО)',
 'посёлок городского типа Октябрьский',                            'Лимассол',
                  'Надеждинский район',             'Калининградская область']
Length: 558, dtype: str

In [148]:
df_hh[df_hh['address'].str.contains(r'\.')]

,title,address,company,short_description,experience,salary
191,Инженер-разработчик,г. Москва,ООО Сенсорика-М,датчики скорости и длины ИСД. - микрометры. Оп...,Опыт 1-3 года,NaN
197,Разработчик C++,г. Москва,ООО Сенсорика-М,Знание языков C / C++. Опыт написания достаточ...,Опыт 3-6 лет,NaN
12423,Продакт-менеджер (направление Посуда),Москва 37.539087,ООО Гауф Рус,Обязательно знание заводов по тематике и работ...,Опыт 3-6 лет,NaN
12535,"Продакт-менеджер по МБТ (кофемашины, термопоты...",Москва 37.539087,ООО Гауф Рус,Опыт запуска продуктов с китайскими заводами. ...,Опыт 3-6 лет,NaN


In [149]:
df_hh['address'] = df_hh['address'].replace({
    'г. Москва':'Москва',
    'Москва 37.539087':'Москва'
})
df_hh['address'].unique()

<StringArray>
[                             'Москва',                          'Зеленоград',
                   'деревня Румянцево',                            'Тольятти',
     'посёлок городского типа Родники',                               'Химки',
                          'Никольское',                         'Красногорск',
                     'Санкт-Петербург',                            'Нью-Йорк',
 ...
    'Черняховский муниципальный округ',                              'Латвия',
                  'посёлок Ольховатка',                             'Испания',
                           'Гвардейск',               'Бугрино (Ненецкий АО)',
 'посёлок городского типа Октябрьский',                            'Лимассол',
                  'Надеждинский район',             'Калининградская область']
Length: 556, dtype: str

In [150]:
towns_regions_countries=pd.read_csv('../towns_regions_contries.csv')
towns_regions_countries.head()

,title,region__title,country_title
0,Москва,Московская область,Россия
1,Санкт-Петербург,Ленинградская область,Россия
2,Новосибирск,Новосибирская область,Россия
3,Екатеринбург,Свердловская область,Россия
4,Нижний Новгород,Нижегородская область,Россия


In [166]:
df_hh_rich=df_hh.merge(towns_regions_countries, how='left', left_on='address',right_on='title')
df_hh_rich.head()

,title_x,address,company,short_description,experience,salary,title_y,region__title,country_title
0,Разработчик,Москва,"Русская Медиагруппа, радиохолдинг",Опыт работы,Опыт 3-6 лет,NaN,Москва,Московская область,Россия
1,Разработчик,Москва,Детский хоспис Дом с маяком,"Уверенное знание Python 3, базовых принципов О...",Опыт 1-3 года,"от 95 000 ₽ за месяц, на руки",Москва,Московская область,Россия
2,Backend-разработчик,Москва,ООО ГК СтиС,2+ года коммерческой разработки на PHP 7.4+. У...,Опыт 1-3 года,NaN,Москва,Московская область,Россия
3,Разработчик ЦФТ,Москва,Bell Integrator,Опыт работы с АБС «ЦФТ-Банк» в качестве,Опыт 3-6 лет,NaN,Москва,Московская область,Россия
4,Разработчик C++,Москва,ООО БУЛАТ,Опыт разработки или исправления/доработки внут...,Опыт 3-6 лет,NaN,Москва,Московская область,Россия


In [167]:
df_hh_rich = df_hh_rich.drop(columns='title_y')


In [168]:
df_hh_rich[df_hh_rich['address'].isin(towns_regions_countries['country_title'])]

,title_x,address,company,short_description,experience,salary,region__title,country_title
2254,Графический дизайнер,Россия,Librico,Опыт работы графическим,Опыт 3-6 лет,"от 80 000 ₽ за месяц, на руки",NaN,NaN
6731,Manual junior+/middle QA engineer,Армения,Ayo Support Solutions,Знание теории тестирования и видов тестировани...,Опыт 1-3 года,NaN,NaN,NaN
7502,QA Engineer (gamedev),Армения,The Open Platform,"More than 2 years of experience in QA, includi...",Опыт 3-6 лет,NaN,NaN,NaN
7547,Junior QA Engineer (Manual),Армения,ТОО Playrix,"Быстрый рост в геймдизайн, продюсирование или ...",Опыт 1-3 года,NaN,NaN,NaN
7566,QA Engineer (Junior/Middle),Армения,Gaijin Games,Опыт тестирования игровых проектов. — Знание с...,Опыт 1-3 года,NaN,NaN,NaN
10974,Product Operations,Латвия,ATAS Ltd,"Характер: готовность напоминать, переспрашиват...",Опыт 1-3 года,NaN,NaN,NaN
11048,Project manager,Латвия,ATAS Ltd,Опыт работы с ATAS или платформами объёмного а...,Опыт 1-3 года,NaN,NaN,NaN


In [169]:
df_hh_rich.loc[2254, 'country_title'] = 'Россия'
df_hh_rich.loc[[6731, 7502,7547,7566], 'country_title'] = 'Армения'
df_hh_rich.loc[[10974, 11048], 'country_title'] = 'Латвия'

## Добавляем признак client_staff_count (кол-во сотрудников компании)

In [155]:
company_staff=pd.read_csv('../company_staff.csv')
company_staff.head()

,firm_name,client_staff_count
0,Филиал ФКУ Налог-Сервис ФНС России по ЦОД в г....,100 — 500
1,Филиал ФКУ Налог-Сервис ФНС России по ЦОД в г....,100 — 500
2,Филиал ФКУ «Налог-Сервис» ФНС России по ЦОД в ...,100 — 500
3,Социальный фонд России,более 5000
4,Duolingo,менее 50


In [170]:
df_hh_rich=df_hh_rich.merge(company_staff, how='left', left_on='company',right_on='firm_name')
df_hh_rich['client_staff_count'].isna().sum()

np.int64(11547)

пупупу.. присоединилось маловато (из 11579). ну ничего, у нас еще много других более полезных признаков.

In [171]:
df_hh_rich.head()

,title_x,address,company,short_description,experience,salary,region__title,country_title,firm_name,client_staff_count
0,Разработчик,Москва,"Русская Медиагруппа, радиохолдинг",Опыт работы,Опыт 3-6 лет,NaN,Московская область,Россия,NaN,NaN
1,Разработчик,Москва,Детский хоспис Дом с маяком,"Уверенное знание Python 3, базовых принципов О...",Опыт 1-3 года,"от 95 000 ₽ за месяц, на руки",Московская область,Россия,NaN,NaN
2,Backend-разработчик,Москва,ООО ГК СтиС,2+ года коммерческой разработки на PHP 7.4+. У...,Опыт 1-3 года,NaN,Московская область,Россия,NaN,NaN
3,Разработчик ЦФТ,Москва,Bell Integrator,Опыт работы с АБС «ЦФТ-Банк» в качестве,Опыт 3-6 лет,NaN,Московская область,Россия,NaN,NaN
4,Разработчик C++,Москва,ООО БУЛАТ,Опыт разработки или исправления/доработки внут...,Опыт 3-6 лет,NaN,Московская область,Россия,NaN,NaN


In [172]:
df_hh_rich=df_hh_rich.drop(columns='firm_name')

# Правим значения в столбце ЗП

Намного удобнее работать с зарплатой как с числом. Поэтому из того вида, в котором сейчас есть этот признак, приведем к числовому. Будем оставлять одно число (от или до, а если указан диапазон - то брать среднее).

In [173]:
df_hh_rich['salary'].unique()

<StringArray>
[                                    nan,
         'от 95 000 ₽ за месяц, на руки',
 '150 000 ₽ за месяц, до вычета налогов',
        'от 200 000 ₽ за месяц, на руки',
        'от 120 000 ₽ за месяц, на руки',
        'до 220 000 ₽ за месяц, на руки',
   '60 000 – 80 000 ₽ за месяц, на руки',
 '250 000 – 350 000 ₽ за месяц, на руки',
        'до 250 000 ₽ за месяц, на руки',
 '150 000 – 200 000 ₽ за месяц, на руки',
 ...
 '300 000 – 400 000 ₽ за месяц, на руки',
 '127 000 – 270 000 ₽ за месяц, на руки',
  '80 000 – 124 410 ₽ за месяц, на руки',
        'до 219 000 ₽ за месяц, на руки',
        'от 225 000 ₽ за месяц, на руки',
          'от 6 000 $ за месяц, на руки',
 '120 000 – 158 000 ₽ за месяц, на руки',
 '140 000 – 180 000 ₽ за месяц, на руки',
 '165 000 ₽ за месяц, до вычета налогов',
 '180 000 – 400 000 ₽ за месяц, на руки']
Length: 1073, dtype: str

In [174]:
import re

def parse_salary(text):
    if pd.isna(text):
        return np.nan
    text=re.sub(r'\s+', '', text)
    nums = re.findall(r'\d+', text)
    nums = list(map(int, nums))
    if len(nums) == 1:
        return nums[0]
    else:
        return sum(nums) / len(nums)
    
df_hh_rich['salary_num'] = df_hh_rich['salary'].apply(parse_salary)
df_hh_rich[~df_hh_rich['salary'].isna()]

,title_x,address,company,short_description,experience,salary,region__title,country_title,client_staff_count,salary_num
1,Разработчик,Москва,Детский хоспис Дом с маяком,"Уверенное знание Python 3, базовых принципов О...",Опыт 1-3 года,"от 95 000 ₽ за месяц, на руки",Московская область,Россия,NaN,95000.0
5,Unity-разработчик,Москва,ООО Нектар,Отлично знаете C#. Знаете классические алгорит...,Опыт более 6 лет,"150 000 ₽ за месяц, до вычета налогов",Московская область,Россия,NaN,150000.0
15,Ведущий разработчик,Москва,БИТ,Владение PL/pgSQL (PostgreSQL) и/или t-sql (MS...,Опыт 1-3 года,"от 200 000 ₽ за месяц, на руки",Московская область,Россия,NaN,200000.0
18,RTL-разработчик,Москва,ООО Трамплин Электроникс,Высшее техническое образование. Опыт разработк...,Опыт 3-6 лет,"от 200 000 ₽ за месяц, на руки",Московская область,Россия,NaN,200000.0
19,Повар-разработчик,Москва,ООО ПРОМЫШЛЕННАЯ КУЛИНАРИЯ,Наличие профессиональных навыков повара. Поним...,Опыт 3-6 лет,"от 120 000 ₽ за месяц, на руки",Московская область,Россия,NaN,120000.0
...,...,...,...,...,...,...,...,...,...,...
10394,Программист/Наладчик станков с ЧПУ (токарь-фре...,Санкт-Петербург,АО СовПлим,Опыт работы на токарном станке с ЧПУ стойка Fa...,Опыт 1-3 года,"130 000 – 170 000 ₽ за месяц, до вычета налогов",Ленинградская область,Россия,NaN,150000.0
10398,"Business development manager (Digital, HoReCa)...",Санкт-Петербург,Restoclub.ru,Работаешь с самыми известными рестораторами ст...,Опыт 1-3 года,"140 000 – 200 000 ₽ за месяц, на руки",Ленинградская область,Россия,NaN,170000.0
10403,Web-программист (full stack разработчик ModX /...,Санкт-Петербург,ИП Федорова Анастасия Сергеевна,Опыт объектно ориентированного программировани...,Опыт 3-6 лет,"120 000 – 160 000 ₽ за месяц, до вычета налогов",Ленинградская область,Россия,NaN,140000.0
10405,R&D химик-разработчик (косметика / beauty / бы...,Санкт-Петербург,Волшебный мир,Опыт работы химиком-,Опыт 3-6 лет,"от 120 000 ₽ за месяц, на руки",Ленинградская область,Россия,NaN,120000.0


# Пропуски в данных

In [177]:
df_hh_rich.isna().sum()

title_x                   0
address                   0
company                   0
short_description       104
experience                0
salary                 8799
region__title          1642
country_title           625
client_staff_count    11547
salary_num             8799
dtype: int64

# Скачиваем датасет

Скачаем датасет и перейдем к соединению двух датасетов и анализу итогового.

In [175]:
df_hh_rich.to_csv('../../analysis/df_hh_rich.csv', index=False)